# RAG Database Setup & Visualization

This notebook contains utilities for:
1. Building the RAG vector database from Ableton manuals
2. Visualizing the vector embeddings

## Import Libraries

In [ ]:
import os
import glob
import numpy as np
from tqdm import tqdm

from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

from sklearn.manifold import TSNE
import plotly.graph_objects as go

## Configuration

In [ ]:
# Embedding Model
EMBED_MODEL = "nomic-embed-text"

# Vector Store Path
DEFAULT_CHROMA_PATH = os.path.expanduser("~/ableton_manual_vectors")
CHROMA_PATH = os.environ.get("CHROMA_PATH", DEFAULT_CHROMA_PATH)

# Path to Ableton Manuals
BASE_PATH = "/Users/mms/github/manuals/*"

## 1. Build RAG Database

### Load Manuals from Directory

In [ ]:
def load_markdown_manuals(base_path):
    """
    Load and split markdown manuals from the specified directory.
    
    Args:
        base_path: Glob pattern for manual directories
    
    Returns:
        List of document chunks with metadata
    """
    include_products = {"live", "push", "move"}
    folders = glob.glob(base_path)
    text_loader_kwargs = {'encoding': 'utf-8'}
    
    documents = []
    for folder in folders:
        product_name = os.path.basename(folder).replace("-manual", "").lower().strip()
        if not any(product in product_name for product in include_products):
            continue

        loader = DirectoryLoader(
            folder, 
            glob="**/*.md", 
            loader_cls=TextLoader, 
            loader_kwargs=text_loader_kwargs
        )
        folder_docs = loader.load()
        
        # Add product metadata to each document
        for doc in folder_docs:
            doc.metadata["product"] = product_name
            documents.append(doc)

    # Split documents into chunks
    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(documents)

    print(f"✅ Loaded {len(chunks)} total chunks from all manuals.")
    return chunks

In [ ]:
# Load documents (only run when needed)
docs = load_markdown_manuals(BASE_PATH)

### Build or Load Vector Store

In [ ]:
def load_or_build_vectorstore(docs=None, persist_directory=CHROMA_PATH, force_rebuild=False):
    """
    Load existing vectorstore if it exists, otherwise build a new one.

    Args:
        docs: Documents to build vectorstore from (only needed if building)
        persist_directory: Path to persist/load the vectorstore
        force_rebuild: If True, rebuild even if vectorstore exists

    Returns:
        vectordb: The loaded or newly built Chroma vectorstore
    """
    embedding = OllamaEmbeddings(model=EMBED_MODEL)

    # Check if vectorstore exists and we're not forcing a rebuild
    if os.path.exists(persist_directory) and not force_rebuild:
        try:
            vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)
            # Verify it has data
            count = vectordb._collection.count()
            if count > 0:
                print(f"✅ Loaded existing vector store with {count:,} vectors.")
                return vectordb
            else:
                print("⚠️  Existing vector store is empty. Rebuilding...")
        except Exception as e:
            print(f"⚠️  Error loading vector store: {e}. Rebuilding...")

    # Build new vectorstore
    if docs is None:
        raise ValueError("Documents are required to build a new vectorstore")

    # Delete existing collection if rebuilding
    if os.path.exists(persist_directory):
        Chroma(persist_directory=persist_directory, embedding_function=embedding).delete_collection()

    vectordb = Chroma.from_documents(
        documents=docs,
        embedding=embedding,
        persist_directory=persist_directory
    )
    print(f"✅ Vector store built and persisted with {len(docs):,} chunks.")
    return vectordb

In [ ]:
# Load existing vectorstore or build new one
# Set force_rebuild=True to rebuild from scratch
vectordb = load_or_build_vectorstore(docs=docs, force_rebuild=False)

## 2. Visualize Vector Embeddings

Visualize the vector embeddings in 2D space using t-SNE dimensionality reduction.

In [ ]:
# Get embedding dimensions
collection = vectordb._collection
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"The vectors have {dimensions:,} dimensions")

In [ ]:
# Retrieve all vectors and metadata
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
doc_types = [metadata['product'] for metadata in result['metadatas']]

# Map products to colors
product_to_color = {'live': 'blue', 'push': 'green', 'move': 'red'}
colors = [product_to_color.get(t, 'gray') for t in doc_types]

print(f"Total vectors: {len(vectors)}")
print(f"Products: {set(doc_types)}")

In [ ]:
# Apply t-SNE dimensionality reduction
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
reduced_vectors = tsne.fit_transform(vectors)

print("✅ t-SNE reduction complete")

In [ ]:
# Create interactive 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Product: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Vector Store Visualization (t-SNE)',
    xaxis_title='t-SNE Component 1',
    yaxis_title='t-SNE Component 2',
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

## Visualize with UMAP

In [ ]:
# Apply UMAP dimensionality reduction
from umap import UMAP

umap_reducer = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
umap_vectors = umap_reducer.fit_transform(vectors)

print("✅ UMAP reduction complete")

In [ ]:
# Create interactive 2D scatter plot with UMAP
fig = go.Figure(data=[go.Scatter(
    x=umap_vectors[:, 0],
    y=umap_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Product: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Vector Store Visualization (UMAP)',
    xaxis_title='UMAP Component 1',
    yaxis_title='UMAP Component 2',
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

## Summary

This notebook provides:
- **Manual Loading**: `load_markdown_manuals()` - Loads and chunks Ableton manuals
- **Vector Store Management**: `load_or_build_vectorstore()` - Builds or loads the RAG database
- **Visualization**: t-SNE plots to visualize embedding clusters by product

The vector store is persisted to disk and can be reused by the chatbot without rebuilding.